In [ ]:
%matplotlib inline

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.signal import freqz
from IPython.display import display, HTML, Math

# ============================================================
# COMPLETE OPTIMIZATION-BASED IIR LOW-PASS DESIGN
# ============================================================

plt.close('all')
plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.io-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.io-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.io-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.45;
    margin-bottom:8px;
}

.io-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:8px;
    font-size:13.5px;
    line-height:1.45;
}

.io-warn{
    width:100%;
    box-sizing:border-box;
    background:#fff8e6;
    border:1px solid #d8b451;
    border-radius:7px;
    padding:9px 11px;
    margin-top:8px;
    font-size:13.5px;
    line-height:1.45;
}

.io-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:4px;
}

.io-code{
    font-family:Consolas,monospace;
    font-size:12.5px;
    line-height:1.45;
    overflow-wrap:anywhere;
}

.io-table{
    border-collapse:collapse;
    font-size:13px;
    width:100%;
}

.io-table th,
.io-table td{
    text-align:center;
    padding:3px 8px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# FILTER SPECIFICATIONS
# ============================================================

fp = 500.0
fsb = 800.0
Fs = 2000.0

Ap_required = 0.25
As_required = 45.0

N = 8
number_of_sections = N//2
number_of_parameters = 4*number_of_sections+1

eps1 = 1e-7
eps2 = 1e-7

p_initial = 2
mu = 2

number_of_attempts = 3

max_outer_iterations = 25
max_inner_iterations = 500

base_random_seed = 12345

nyquist = Fs/2.0

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML(f"""
<div class="io-root">

<div class="io-header">
Complete Optimization-Based IIR Low-Pass Design
</div>

<div class="io-doc">

This notebook performs the complete optimization procedure starting only from
the filter specifications. <b>No coefficients from a previously known solution
are used.</b>

<div style="text-align:center;font-size:14.5px;margin:6px 0;">
<b>
f<sub>p</sub> = {fp:.0f} Hz,&nbsp;&nbsp;
f<sub>s</sub> = {fsb:.0f} Hz,&nbsp;&nbsp;
F<sub>s</sub> = {Fs:.0f} Hz,&nbsp;&nbsp;
A<sub>p</sub> = {Ap_required:.2f} dB,&nbsp;&nbsp;
A<sub>s</sub> = {As_required:.0f} dB,&nbsp;&nbsp;
N = {N}.
</b>
</div>

For N = {N}, the filter is represented as four second-order sections and the
optimization vector contains {number_of_parameters} unknown parameters:

<div style="text-align:center;font-size:14px;margin:6px 0;">
<b>
ξ = [α₀₁, α₁₁, β₀₁, β₁₁, ..., α₀₄, α₁₄, β₀₄, β₁₄, H₀].
</b>
</div>

The initial vector x₀ is generated randomly. A dense frequency grid containing
20 times as many samples as unknown parameters is constructed over the passband
and stopband. The transition band is excluded from the optimization.

The weighted approximation error is

<div style="text-align:center;font-size:14px;margin:6px 0;">
<b>
e<sub>i</sub>(x) = w<sub>i</sub>[M(x,ω<sub>i</sub>) - M₀(ω<sub>i</sub>)].
</b>
</div>

The minimax solution is approached by a sequence of least-p problems.
The value of p begins at 2 and is doubled after each outer iteration.

The number of outer iterations is therefore calculated by the algorithm; it is
not prescribed in advance. Several independent random starting vectors are
tried. If none of the resulting filters satisfies all specifications, the
solution with the smallest maximum weighted error is retained.

Finally, unstable poles are reflected inside the unit circle, the normalization
constant is corrected automatically, and the final eighth-order transfer
function is constructed symbolically with SymPy.

<div class="io-warn">

<b>Implementation note.</b>
This notebook is an independent Python implementation of the optimization-based
IIR design procedure. The numerical results are not expected to reproduce the
MATLAB example coefficient by coefficient.

The original implementation uses MATLAB <b>fminunc</b> with its own optimization
strategy and a randomly initialized parameter vector, whereas this notebook
uses SciPy's <b>BFGS quasi-Newton optimizer</b>, a reproducible random
initialization, and a least-p sequence with progressively increasing values of p.

Consequently, the number of iterations, the optimized parameter vector, and the
final filter coefficients may differ, although both implementations solve the
same design problem and are evaluated against the same frequency-domain
specifications.

<br><br>

The purpose of this notebook is therefore not to reproduce a particular MATLAB
run, but to demonstrate computationally the complete optimization process from
the design specifications to the final stable IIR filter.

</div>

</div>

</div>
"""))

# ============================================================
# FREQUENCY GRID
# ============================================================

number_of_frequency_samples = 20*number_of_parameters

active_bandwidth = fp+(nyquist-fsb)

frequency_step = active_bandwidth/(number_of_frequency_samples-2)

number_passband = int(np.floor(fp/frequency_step+0.5))+1
number_stopband = number_of_frequency_samples-number_passband

frequency_passband = np.linspace(0.0,fp,number_passband)
frequency_stopband = np.linspace(fsb,nyquist,number_stopband)

frequency_grid = np.concatenate((frequency_passband,frequency_stopband))

omega_grid = 2.0*np.pi*frequency_grid/Fs

desired_magnitude = np.concatenate((np.ones(number_passband),np.zeros(number_stopband)))

epsilon_p = (10.0**(0.05*Ap_required)-1.0)/(10.0**(0.05*Ap_required)+1.0)
epsilon_s = 10.0**(-0.05*As_required)

stopband_weight = epsilon_p/epsilon_s

weight_grid = np.concatenate((np.ones(number_passband),stopband_weight*np.ones(number_stopband)))

# ============================================================
# FILTER MODEL
# ============================================================

def model_magnitude(x,omega):

    z = np.exp(1j*omega)

    H = np.full_like(z,x[-1],dtype=complex)

    for section in range(number_of_sections):

        i = 4*section

        alpha0 = x[i]
        alpha1 = x[i+1]
        beta0 = x[i+2]
        beta1 = x[i+3]

        numerator = alpha0+alpha1*z+z**2
        denominator = beta0+beta1*z+z**2

        H *= numerator/denominator

    return np.abs(H)

# ============================================================
# WEIGHTED ERROR
# ============================================================

def weighted_error(x):

    M = model_magnitude(x,omega_grid)

    return weight_grid*(M-desired_magnitude)

# ============================================================
# LEAST-p OBJECTIVE
# ============================================================

def least_p_objective(x,p):

    e = np.abs(weighted_error(x))

    Emax = np.max(e)

    if not np.isfinite(Emax):

        return 1e100

    if Emax == 0.0:

        return 0.0

    normalized_error = e/Emax

    return Emax*np.sum(normalized_error**p)**(1.0/p)

# ============================================================
# RANDOM INITIAL VECTOR
# ============================================================

def generate_initial_vector(seed):

    rng = np.random.default_rng(seed)

    x0 = np.empty(number_of_parameters)

    x0[:-1] = rng.uniform(0.05,1.50,number_of_parameters-1)
    x0[-1] = rng.uniform(0.10,1.00)

    return x0

# ============================================================
# AUTOMATIC STABILITY CORRECTION
# ============================================================

def stabilize_solution(x):

    x_stable = x.copy()

    H0_stable = float(x[-1])

    stability_data = []

    for section in range(number_of_sections):

        i = 4*section

        beta0 = float(x[i+2])
        beta1 = float(x[i+3])

        poles = np.roots([1.0,beta1,beta0])

        original_maximum_radius = np.max(np.abs(poles))

        corrected_poles = []

        reflected = 0

        for pole in poles:

            if abs(pole) > 1.0:

                corrected_pole = 1.0/np.conj(pole)

                H0_stable /= abs(pole)

                corrected_poles.append(corrected_pole)

                reflected += 1

            else:

                corrected_poles.append(pole)

        corrected_polynomial = np.real_if_close(np.poly(corrected_poles),tol=1000)

        corrected_polynomial = np.asarray(corrected_polynomial,dtype=float)

        x_stable[i+2] = corrected_polynomial[2]
        x_stable[i+3] = corrected_polynomial[1]

        final_poles = np.roots([1.0,x_stable[i+3],x_stable[i+2]])

        final_maximum_radius = np.max(np.abs(final_poles))

        stability_data.append((section+1,original_maximum_radius,reflected,final_maximum_radius))

    x_stable[-1] = H0_stable

    return x_stable,stability_data

# ============================================================
# CONVERSION TO POLYNOMIAL COEFFICIENTS
# ============================================================

def transfer_function_coefficients(x):

    b = np.array([x[-1]],dtype=float)
    a = np.array([1.0],dtype=float)

    for section in range(number_of_sections):

        i = 4*section

        numerator = np.array([1.0,x[i+1],x[i]],dtype=float)
        denominator = np.array([1.0,x[i+3],x[i+2]],dtype=float)

        b = np.convolve(b,numerator)
        a = np.convolve(a,denominator)

    return b,a

# ============================================================
# FILTER PERFORMANCE
# ============================================================

def measure_filter(x):

    b,a = transfer_function_coefficients(x)

    frequency = np.linspace(0.0,nyquist,12000)

    omega = 2.0*np.pi*frequency/Fs

    _,H = freqz(b,a,worN=omega)

    magnitude = np.abs(H)

    magnitude_db = 20.0*np.log10(np.maximum(magnitude,1e-15))

    passband_mask = frequency <= fp
    stopband_mask = frequency >= fsb

    maximum_passband_deviation = np.max(np.abs(magnitude_db[passband_mask]))

    passband_peak_to_peak = np.max(magnitude_db[passband_mask])-np.min(magnitude_db[passband_mask])

    stopband_attenuation = -np.max(magnitude_db[stopband_mask])

    return frequency,magnitude,magnitude_db,maximum_passband_deviation,passband_peak_to_peak,stopband_attenuation,b,a

# ============================================================
# ONE COMPLETE OPTIMIZATION ATTEMPT
# ============================================================

def run_attempt(seed):

    x = generate_initial_vector(seed)

    p = p_initial

    history = []

    previous_maximum_error = None

    for outer_iteration in range(1,max_outer_iterations+1):

        x_previous = x.copy()

        result = minimize(lambda v:least_p_objective(v,p),x,method='BFGS',options={'gtol':eps2,'maxiter':max_inner_iterations})

        x = result.x.copy()

        maximum_error = np.max(np.abs(weighted_error(x)))

        delta_x = np.linalg.norm(x-x_previous)

        objective_value = least_p_objective(x,p)

        history.append({'iteration':outer_iteration,'p':p,'inner_iterations':result.nit,'objective':objective_value,'maximum_error':maximum_error,'delta_x':delta_x})

        if previous_maximum_error is not None:

            if abs(previous_maximum_error-maximum_error) < eps1:

                break

        previous_maximum_error = maximum_error

        p *= mu

    maximum_error = np.max(np.abs(weighted_error(x)))

    x_stable,stability_data = stabilize_solution(x)

    frequency,magnitude,magnitude_db,passband_deviation,passband_peak_to_peak,stopband_attenuation,b,a = measure_filter(x_stable)

    specifications_satisfied = passband_deviation <= Ap_required and stopband_attenuation >= As_required

    return {
        'seed':seed,
        'x_raw':x,
        'x_stable':x_stable,
        'history':history,
        'maximum_error':maximum_error,
        'stability_data':stability_data,
        'frequency':frequency,
        'magnitude':magnitude,
        'magnitude_db':magnitude_db,
        'passband_deviation':passband_deviation,
        'passband_peak_to_peak':passband_peak_to_peak,
        'stopband_attenuation':stopband_attenuation,
        'b':b,
        'a':a,
        'specifications_satisfied':specifications_satisfied
    }

# ============================================================
# RUN THE INDEPENDENT OPTIMIZATION ATTEMPTS
# ============================================================

attempts = []

for attempt_index in range(number_of_attempts):

    seed = base_random_seed+attempt_index

    attempts.append(run_attempt(seed))

# ============================================================
# SELECT THE BEST ATTEMPT
# ============================================================

successful_attempts = [attempt for attempt in attempts if attempt['specifications_satisfied']]

if len(successful_attempts) > 0:

    selected = min(successful_attempts,key=lambda attempt:attempt['maximum_error'])

else:

    selected = min(attempts,key=lambda attempt:attempt['maximum_error'])

selected_attempt_number = attempts.index(selected)+1

# ============================================================
# OPTIMIZATION ATTEMPTS TABLE
# ============================================================

attempt_rows = ""

for index,attempt in enumerate(attempts,1):

    attempt_rows += f"""
    <tr>
    <td>{index}</td>
    <td>{attempt['seed']}</td>
    <td>{len(attempt['history'])}</td>
    <td>{attempt['maximum_error']:.3e}</td>
    <td>{attempt['passband_deviation']:.4f} dB</td>
    <td>{attempt['stopband_attenuation']:.2f} dB</td>
    <td><b>{"PASS" if attempt['specifications_satisfied'] else "FAIL"}</b></td>
    </tr>
    """

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">Optimization attempts</div>

<table class="io-table">

<tr>
<th>Attempt</th>
<th>Seed</th>
<th>Outer iterations</th>
<th>Max weighted error</th>
<th>Passband deviation</th>
<th>Stopband attenuation</th>
<th>Specs</th>
</tr>

{attempt_rows}

</table>

<div style="margin-top:6px;">

Selected attempt: <b>{selected_attempt_number}</b>

&nbsp;&nbsp;&nbsp;

Frequency samples: <b>{number_of_frequency_samples}</b>

&nbsp;&nbsp;&nbsp;

Passband samples: <b>{number_passband}</b>

&nbsp;&nbsp;&nbsp;

Stopband samples: <b>{number_stopband}</b>

</div>

</div>

</div>
"""))

# ============================================================
# CALCULATED OPTIMIZATION VECTOR
# ============================================================

xi = selected['x_raw']

xi_text = "["+", ".join(f"{value:.6f}" for value in xi)+"]"

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">
Calculated optimization vector ξ
</div>

<div class="io-code">
{xi_text}
</div>

<div style="margin-top:5px;">
This vector was calculated by the Python optimization procedure; it was not
entered as input data.
</div>

</div>

</div>
"""))

# ============================================================
# LEAST-p ITERATION HISTORY
# ============================================================

history_rows = ""

for record in selected['history']:

    history_rows += f"""
    <tr>
    <td>{record['iteration']}</td>
    <td>{record['p']}</td>
    <td>{record['inner_iterations']}</td>
    <td>{record['objective']:.3e}</td>
    <td>{record['maximum_error']:.3e}</td>
    <td>{record['delta_x']:.3e}</td>
    </tr>
    """

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">
Least-p iteration history of the selected attempt
</div>

<table class="io-table">

<tr>
<th>k</th>
<th>p</th>
<th>BFGS iterations</th>
<th>Ψ<sub>p</sub></th>
<th>max |e|</th>
<th>||δx||</th>
</tr>

{history_rows}

</table>

</div>

</div>
"""))

# ============================================================
# AUTOMATIC STABILITY CORRECTION TABLE
# ============================================================

stability_rows = ""

for section,original_radius,reflected,final_radius in selected['stability_data']:

    stability_rows += f"""
    <tr>
    <td>H<sub>{section}</sub></td>
    <td>{original_radius:.6f}</td>
    <td>{reflected}</td>
    <td>{final_radius:.6f}</td>
    </tr>
    """

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">
Automatic stability correction
</div>

<table class="io-table">

<tr>
<th>Section</th>
<th>Original maximum pole radius</th>
<th>Reflected poles</th>
<th>Final maximum pole radius</th>
</tr>

{stability_rows}

</table>

<div style="margin-top:6px;">

H₀ before correction:
<b>{selected['x_raw'][-1]:.6f}</b>

&nbsp;&nbsp;&nbsp;

H₀ after correction:
<b>{selected['x_stable'][-1]:.6f}</b>

</div>

</div>

</div>
"""))

# ============================================================
# FINAL SECOND-ORDER SECTIONS
# ============================================================

x_stable = selected['x_stable']

section_rows = ""

for section in range(number_of_sections):

    i = 4*section

    section_rows += f"""
    <tr>
    <td>H<sub>{section+1}</sub>(z)</td>
    <td>{x_stable[i]:.6f}</td>
    <td>{x_stable[i+1]:.6f}</td>
    <td>{x_stable[i+2]:.6f}</td>
    <td>{x_stable[i+3]:.6f}</td>
    </tr>
    """

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">
Stable second-order sections
</div>

<table class="io-table">

<tr>
<th>Section</th>
<th>α₀</th>
<th>α₁</th>
<th>β₀</th>
<th>β₁</th>
</tr>

{section_rows}

</table>

</div>

</div>
"""))

# ============================================================
# SYMBOLIC CONSTRUCTION OF THE FINAL TRANSFER FUNCTION
# ============================================================

q = sp.symbols('q')

H_symbolic = sp.Float(x_stable[-1])

for section in range(number_of_sections):

    i = 4*section

    numerator = 1.0+x_stable[i+1]*q+x_stable[i]*q**2
    denominator = 1.0+x_stable[i+3]*q+x_stable[i+2]*q**2

    H_symbolic *= numerator/denominator

H_symbolic = sp.cancel(H_symbolic)

numerator_symbolic,denominator_symbolic = sp.fraction(H_symbolic)

numerator_symbolic = sp.Poly(sp.expand(numerator_symbolic),q)
denominator_symbolic = sp.Poly(sp.expand(denominator_symbolic),q)

b_symbolic = np.array([float(value) for value in reversed(numerator_symbolic.all_coeffs())])
a_symbolic = np.array([float(value) for value in reversed(denominator_symbolic.all_coeffs())])

a0_symbolic = a_symbolic[0]

b_symbolic = b_symbolic/a0_symbolic
a_symbolic = a_symbolic/a0_symbolic

# ============================================================
# FINAL COEFFICIENTS
# ============================================================

b = selected['b']
a = selected['a']

numerator_text = "["+", ".join(f"{value:.6f}" for value in b)+"]"
denominator_text = "["+", ".join(f"{value:.6f}" for value in a)+"]"

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">
Final eighth-order transfer function coefficients
</div>

<b>Numerator b:</b>

<div class="io-code">
{numerator_text}
</div>

<br>

<b>Denominator a:</b>

<div class="io-code">
{denominator_text}
</div>

</div>

</div>
"""))

# ============================================================
# FINAL SPECIFICATION CHECK
# ============================================================

passband_deviation = selected['passband_deviation']
passband_peak_to_peak = selected['passband_peak_to_peak']
stopband_attenuation = selected['stopband_attenuation']

passband_status = "PASS" if passband_deviation <= Ap_required else "FAIL"
stopband_status = "PASS" if stopband_attenuation >= As_required else "FAIL"

display(HTML(f"""
<div class="io-root">

<div class="io-box">

<div class="io-title">
Final specification check
</div>

<table class="io-table">

<tr>
<th>Quantity</th>
<th>Calculated value</th>
<th>Requirement</th>
<th>Result</th>
</tr>

<tr>
<td>Maximum passband deviation</td>
<td>{passband_deviation:.6f} dB</td>
<td>≤ {Ap_required:.2f} dB</td>
<td><b>{passband_status}</b></td>
</tr>

<tr>
<td>Passband peak-to-peak ripple</td>
<td>{passband_peak_to_peak:.6f} dB</td>
<td>—</td>
<td>—</td>
</tr>

<tr>
<td>Stopband attenuation</td>
<td>{stopband_attenuation:.6f} dB</td>
<td>≥ {As_required:.0f} dB</td>
<td><b>{stopband_status}</b></td>
</tr>

</table>

</div>

</div>
"""))

# ============================================================
# DATA FOR THE FINAL FIGURE
# ============================================================

frequency = selected['frequency']
magnitude = selected['magnitude']
magnitude_db = selected['magnitude_db']

passband_mask = frequency <= fp
stopband_mask = frequency >= fsb

all_poles = np.roots(a)
all_zeros = np.roots(b)

# ============================================================
# FINAL FIGURE — BINDER-SAFE INLINE RENDERING
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.5))

ax1,ax2,ax3,ax4 = axes.flat

# ------------------------------------------------------------
# 1. COMPLETE MAGNITUDE RESPONSE
# ------------------------------------------------------------

ax1.plot(frequency,magnitude_db,linewidth=1.3)

ax1.axvline(fp,linestyle='--',linewidth=0.9)
ax1.axvline(fsb,linestyle='--',linewidth=0.9)
ax1.axhline(-As_required,linestyle=':',linewidth=0.9)

ax1.set_xlim(0,nyquist)
ax1.set_ylim(-120,5)

ax1.set_title('Complete Magnitude Response')
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('Magnitude (dB)')

ax1.grid(True,linestyle=':',alpha=0.25)

# ------------------------------------------------------------
# 2. PASSBAND DETAIL
# ------------------------------------------------------------

ax2.plot(frequency[passband_mask],magnitude_db[passband_mask],linewidth=1.3)

ax2.axhline(Ap_required,linestyle=':',linewidth=0.9)
ax2.axhline(-Ap_required,linestyle=':',linewidth=0.9)

ax2.set_xlim(0,fp)
ax2.set_ylim(-0.30,0.30)

ax2.set_title('Passband Detail')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Magnitude (dB)')

ax2.grid(True,linestyle=':',alpha=0.25)

# ------------------------------------------------------------
# 3. STOPBAND DETAIL
# ------------------------------------------------------------

ax3.plot(frequency[stopband_mask],magnitude_db[stopband_mask],linewidth=1.3)

ax3.axhline(-As_required,linestyle=':',linewidth=0.9)

ax3.set_xlim(fsb,nyquist)
ax3.set_ylim(-120,-35)

ax3.set_title('Stopband Detail')
ax3.set_xlabel('Frequency (Hz)')
ax3.set_ylabel('Magnitude (dB)')

ax3.grid(True,linestyle=':',alpha=0.25)

# ------------------------------------------------------------
# 4. POLE-ZERO DIAGRAM
# ------------------------------------------------------------

theta = np.linspace(0,2*np.pi,500)

ax4.plot(np.cos(theta),np.sin(theta),'--',linewidth=0.9)

ax4.plot(np.real(all_zeros),np.imag(all_zeros),'o',fillstyle='none',markersize=6,label='Zeros')
ax4.plot(np.real(all_poles),np.imag(all_poles),'x',markersize=6,label='Poles')

ax4.axhline(0,linewidth=0.7)
ax4.axvline(0,linewidth=0.7)

ax4.set_xlim(-1.2,1.2)
ax4.set_ylim(-1.2,1.2)

ax4.set_aspect('equal',adjustable='box')

ax4.set_title('Pole-Zero Diagram')
ax4.set_xlabel('Real part')
ax4.set_ylabel('Imaginary part')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.28,hspace=0.46)

# ============================================================
# INLINE DISPLAY — NO ipympl CANVAS
# ============================================================

plt.show()

plt.close(fig)